In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [5]:
# 1. dataset loading

"""
x_new = (x - 0.1307) / 0.3081
x normalized formula
"""
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data',
                               train=True,
                               download=True,
                               transform=transform)
test_dataset = datasets.MNIST(root='./data',
                              train=False,
                              download=True,
                              transform=transform)

#shuffle randomize the train dataset

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)


100%|██████████| 9.91M/9.91M [00:00<00:00, 18.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 447kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.26MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.19MB/s]


In [6]:
# 2. model

class MNISTNet(nn.Module):
    def __init__(self):
        super(MNISTNet, self).__init__()

        """
        A linear layer computes: Y = X * W^T + b
        • Weight matrix parameters: 300 * 784 = 235,200 weights
        • Bias parameters: 300 biases
        • Total parameters: 235,200 + 300 = 235,500 parameters

        """

        self.fc1 = nn.Linear(28 * 28, 300)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(300, 100)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(100, 10)

    def forward(self, x):
      """
      Original image shape: 28 x 28
      But Linear expects a vector.
      So: 28 x 28 -> 784

      For a batch: 64 x 1 x 28 x 28 becomes 64 x 784

      What does -1 mean?
      x.view(-1, 784) means: PyTorch, calculate the first dimension automatically.

      If there are 64 images: 64 x 784
      If there are 100 images: 100 x 784
      """
      x = x.view(-1, 28 * 28)
      x = self.relu1(self.fc1(x))
      x = self.relu2(self.fc2(x))
      return self.fc3(x)







In [8]:
# 3. Training and eval

def train_epoch(model, loader, criterion, optimizer, device):
  model.train()
  for data, target in loader:
    data, target = data.to(device), target.to(device)
    # resets accumulated gradients to zero
    optimizer.zero_grad()
    output = model(data)
    loss = criterion(output, target)
    #calculate the gradient for movement direction to reduce the weight
    loss.backward()
    #uses gradient to update the weight
    optimizer.step()

def evaluate(model, loader, device):
  model.eval()
  correct = 0
  total = 0

  # we dont want grad learning during the eval
  with torch.no_grad():
    for data, target in loader:
      data, target = data.to(device), target.to(device)
      output = model(data)
      #argmax finds the position of the largest value.
      pred = output.argmax(dim=1, keepdim=True)
      #count pred: 7 actual=7 then correct
      correct += pred.eq(target.view_as(pred)).sum().item()
      total += target.size(0)


  """
  Accuracy = (correct predictions / total predictions) * 100
  """
  return 100.0 * correct / total


In [14]:
# 4. magnitude pruning and hook

def apply_magnitude_pruning(model, sparsity_ratio):
  """
    1. Collects weight tensors across the network.
    2. Calculates absolute magnitude threshold using kthvalue.
    3. Creates binary masks (1 = keep, 0 = prune).
    4. Masks parameter values and attaches backward hooks to keep pruned
     weights at zero.
    """

  for name, param in model.named_parameters():
    if 'weight' in name:
      """
      for these:

      fc1.weight
      fc1.bias
      fc2.weight
      fc2.bias
      fc3.weight
      fc3.bias
      """

      num_elements = param.numel()

      #elements we need to prune
      """
      Suppose:
      num_elements = 1000
      and:
      sparsity = 0.80
      Then:
      k = 1000 * 0.8
      k = 800
      So we're trying to identify approximately 800 small weights.
      """
      k = int(num_elements * sparsity_ratio)

      if k >0 and k < num_elements:
        # convert the params to abs and set the matrix to one long vector and find the kth item
        threshold = torch.kthvalue(
        param.abs().view(-1),
        k
          ).values.item()

        # Create binary mask
        mask = (param.abs() > threshold).float()

        """
        weights:
        [0.01, 0.03, 0.8, 1.2]

        mask:

        [0, 0, 1, 1]
        """

        # Zero out pruned weights
        param.data.mul_(mask)

        """
        The hook does:
        grad_new = grad * mask
        If:
        mask = 0
        then:
        grad_new = 0
        Therefore the optimizer can't update that weight through a gradient.
        If:
        mask = 1
        then:
        grad_new = grad
        The weight continues learning normally.
        """
        #beacuse the weight that we are trying to update could become 0
        param.register_hook(lambda grad, m=mask: grad * m)





In [15]:
# 5. execution
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MNISTNet().to(device)
"""
For classification, you use cross-entropy loss.
The model outputs logits:
z_1, z_2, ..., z_10
Cross entropy compares these with the correct class.
Conceptually:
high probability for correct class -> low loss
"""
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-3)

#A: Baseline model
print("--- Training Baseline Model ---")

for epoch in range(1, 6):

    train_epoch(model, train_loader, criterion, optimizer, device)

    acc = evaluate(model, test_loader, device)

    print(f"Epoch {epoch}: Accuracy = {acc:.2f}%")

baseline_acc = evaluate(model, test_loader, device)
print(f"\nBaseline Model Accuracy: {baseline_acc:.2f}%")

#B: Prune Model (80% sparsity)
sparsity = 0.80
print(f"\n--- Applying {int(sparsity * 100)}% Magnitude Pruning ---")
apply_magnitude_pruning(model, sparsity)
pruned_acc = evaluate(model, test_loader, device)

print(f"Accuracy immediately after pruning (No Fine-tuning): {pruned_acc:.2f}%")

#C: Fine-tune remaining non-zero weights for 2 epochs
print("\n--- Fine-Tuning Remaining Weights (2 Epochs) ---")

"""
Reduced the learning rate because of sparsity
Optimizer will update the weights and reduce the loss
"""
fine_tune_optimizer = optim.Adam(model.parameters(), lr=1e-4)
for ft_epoch in range(1, 3):

    train_epoch(
        model,
        train_loader,
        criterion,
        fine_tune_optimizer,
        device
    )

    ft_acc = evaluate(model, test_loader, device)

    print(f"Fine-tuning Epoch {ft_epoch}: Accuracy = {ft_acc:.2f}%")



--- Training Baseline Model ---
Epoch 1: Accuracy = 96.40%
Epoch 2: Accuracy = 97.20%
Epoch 3: Accuracy = 97.62%
Epoch 4: Accuracy = 97.73%
Epoch 5: Accuracy = 97.77%

Baseline Model Accuracy: 97.77%

--- Applying 80% Magnitude Pruning ---
Accuracy immediately after pruning (No Fine-tuning): 91.87%

--- Fine-Tuning Remaining Weights (2 Epochs) ---
Fine-tuning Epoch 1: Accuracy = 97.97%
Fine-tuning Epoch 2: Accuracy = 98.15%
